# DiD : IRA & CHIPS Act et construction d'usines

Question : quelle part du boom de la construction d'usines électronique / électrique est causée par les lois d'août 2022 ?
On compare ce secteur aux 6 autres sous-secteurs manufacturiers, en log des dépenses réelles (M\$ de 2019). Les calculs sont dans [`src/did.py`](../src/did.py).

In [1]:
import sys

import polars as pl

sys.path.append('../src')
import data_processing as dp
import did
from plot_functions import set_theme, plot_event_study, plot_placebos, plot_specs

set_theme()
PANEL = dp.load_processed()['did_panel_monthly']
MFG = PANEL.filter(pl.col('sector').str.starts_with('mfg_'))

## 1. Le design

![Graphe causal du DiD](../reports/figures/dag_did.png)

- **Ce que le DiD retire** : les différences stables entre secteurs (α : taille, intensité en capital) et les chocs communs à tous (λ : taux de la Fed, prix des matériaux).
- **Ce qu'il ne retire pas** : un facteur qui évolue différemment selon le secteur (U), comme la pénurie de puces de 2021 ou la demande liée à l'IA.
- **Hypothèses du cours** : *consistency* (le traitement est le paquet CHIPS + IRA), *no anticipation* (rien ne réagit avant les lois) et *tendances parallèles* (sans les lois, le traité aurait évolué comme les contrôles).

## 2. Le DiD 2×2

In [2]:
did.cell_means(MFG).with_columns(pl.exclude('groupe').round(3))

groupe,avant,après,variation
str,f64,f64,f64
"""Traité""",9.036,11.043,2.007
"""Contrôles""",8.273,8.571,0.298
"""Écart traité − contrôles""",0.763,2.472,1.709


Avant = juillet 2019 → juin 2021, après = août 2022 → août 2026 ; les 13 mois de transition entre les deux sont exclus (section 3). Le traité gagne **2,007** en log, les contrôles **0,298** : δ = **1,709**.
La régression TWFE sur les mêmes mois, $Y_{st} = \alpha_s + \lambda_t + \delta\, T_{st} + \varepsilon_{st}$, donne le même δ :

In [3]:
did.twfe(MFG)

{'delta': 1.7087749555144292, 'se_cluster': 0.09395508484994615}

δ = **1,709**, identique au 2×2. Son erreur standard clusterisée par secteur (0,094) n'est pas fiable avec un seul secteur traité : la section 5 juge l'incertitude autrement.

## 3. Pré-tendances

In [4]:
es = did.event_study(MFG)
plot_event_study(es, did.BEFORE, did.TRANSITION,
                 title='Électronique / électrique vs contrôles, trimestre par trimestre',
                 subtitle="Écart de log, 0 = écart moyen de juillet 2019 à juin 2021")

Avant les lois, l'écart passe par trois phases : un **rattrapage** de 2017 à 2019 (−2,05 en 2017-T1), un **plateau** de mi-2019 à mi-2021 (entre −0,15 et +0,13), puis une **hausse dès 2021-T4** (+0,60).
On prend donc le plateau comme période avant et on exclut la transition, qui peut venir de l'anticipation ou de la pénurie de puces : une fenêtre depuis 2015 mélangerait le rattrapage avec l'effet.
Ce plateau a été choisi en regardant l'event study : la section 6 montre la fenêtre depuis 2015 pour mesurer ce que ce choix change.

## 4. L'effet estimé

In [5]:
did.readings(MFG).with_columns(pl.col('delta', 'effet').round(3))

lecture,avant,delta,effet
str,str,f64,f64
"""Borne haute : anticipation""","""2019-07 → 2021-06""",1.709,4.522
"""Borne basse : pénurie de puces""","""2021-07 → 2022-07""",1.061,1.89


La **borne haute** suppose que, sans les lois, le secteur serait resté au niveau du plateau 2019-2021 : toute la hausse de fin 2021 est de l'anticipation, et l'effet vaut **+452 %**.
La **borne basse** suppose que la hausse de la transition, poussée par la pénurie de puces, aurait eu lieu sans les lois : elle sert de référence, et l'effet tombe à **+189 %**.
Les données ne permettent pas de trancher entre les deux.

In [6]:
did.dollars(MFG).with_columns(pl.col('effet').round(3), pl.col('md_par_an').round(1))

borne,effet,md_par_an
str,f64,f64
"""Borne haute : anticipation""",4.522,53.9
"""Borne basse : pénurie de puces""",1.89,43.0


Concrètement, les lois ajouteraient **43,0 à 53,9 Md\$ de 2019 par an** aux dépenses de construction d'usines du secteur depuis août 2022.
L'écart entre les bornes est plus faible en dollars qu'en % : les deux partent de la même dépense observée, seule la part contrefactuelle change.

In [7]:
did.dynamics(es).with_columns(pl.col('coef', 'effet').round(3))

moment,trimestre,coef,effet
str,str,f64,f64
"""Premier trimestre""","""2022-T3""",1.26,2.527
"""Pic""","""2023-T4""",2.066,6.894
"""Dernier trimestre""","""2026-T3""",1.167,2.214


L'effet monte de **+253 %** en 2022-T3 à un **pic de +689 % en 2023-T4**, puis retombe à **+221 %** en 2026-T3 (juillet-août).
C'est attendu pour un flux de dépenses de construction : la vague de chantiers retombe quand les usines sont livrées.
Le pic et la retombée sont mesurés par rapport au plateau : ils se lisent comme la borne haute.

## 5. Est-ce plus qu'une fluctuation ?

In [8]:
plot_placebos(did.time_placebos(MFG), did.did_2x2(MFG),
              title='Le vrai δ face aux DiD placebo dans le temps',
              subtitle='Même DiD 2×2 (24 mois avant, 13 exclus, 49 après) sur de fausses dates')

Aucun des **53** placebos n'atteint le vrai δ, le plus grand vaut **0,66** ; les fausses dates s'arrêtent en juin 2017 pour que la fenêtre après ne déborde pas sur la hausse de 2021.
Décalées d'un mois, avec des fenêtres de 24 et 49 mois qui se recouvrent presque entièrement, elles ne sont pas des tirages indépendants : « 0 / 53 » n'est pas une p-valeur, ce qui compte est l'écart entre le vrai δ et le plus grand placebo.
Cet écart passe de **1,05** (δ = 1,709) à **0,40** avec la borne basse (δ = 1,061) : le vrai δ reste au-dessus, avec une marge bien plus faible.

In [9]:
did.permutation(MFG).with_columns(pl.all().round(3))

delta_vrai,placebo_min,placebo_max,p
f64,f64,f64,f64
1.709,-0.677,-0.089,0.143


Entre secteurs (« replay the lottery »), on attribue le traitement à chacun des 7 secteurs : le vrai δ (**1,709**) est le seul positif, les placebos vont de −0,677 à −0,089.
La p-valeur vaut **0,143**, soit 1/7 : c'est le minimum possible avec 7 secteurs.

## 6. Robustesse

In [10]:
plot_specs(did.robustness(PANEL), 'spécification', 'effet', 'Principale (hors transition)',
           title="L'effet selon la spécification", subtitle='Effet moyen depuis août 2022, exp(δ) − 1',
           xtitle='Effet estimé')

L'effet reste entre **+430 %** et **+495 %** sans l'équipement de transport (usines de véhicules électriques) ou avec bureaux et commerces en contrôles.
En niveaux (**+591 %**), on suppose que le traité aurait gagné autant de dollars que les contrôles plutôt que le même pourcentage ; on garde les logs, car la question porte sur un pourcentage.
La fenêtre depuis 2015 donne **+915 %** parce qu'elle mélange le rattrapage 2017-2019 avec l'effet.

## 7. À retenir

- **Effet** : entre **+189 %** (borne basse, pénurie de puces) et **+452 %** (borne haute, anticipation) depuis août 2022, soit **43,0 à 53,9 Md\$ de 2019 par an**.
- **Dynamique** : pic à +689 % en 2023-T4, puis +221 % mi-2026 (mesurés depuis le plateau, donc borne haute), comme une vague de chantiers.
- **Pré-tendances** : plateau de mi-2019 à mi-2021, choisi sur l'event study, puis hausse dès fin 2021 ; anticipation ou pénurie de puces, les données ne tranchent pas, d'où la fourchette.
- **Incertitude** : pour les deux bornes, le vrai δ dépasse le plus grand placebo dans le temps (0,66) ; entre secteurs, p = 1/7, le minimum possible.
- **Robustesse** : stable selon les contrôles (+430 % à +495 %), très sensible à la fenêtre avant (+915 % depuis 2015).
- **Limite** : un seul secteur traité et un boom de l'IA simultané, que le DiD ne sépare pas des lois.